In [24]:
import pandas as pd
import psycopg2
import requests
from sklearn.svm import SVC
import mlflow
import os
from sklearn.metrics import accuracy_score, mean_absolute_error,r2_score

IP_MLFLOW = "http://10.43.101.168:30500"
API_URL = "http://10.43.101.108/data"
REGEN_URL = "http://10.43.101.108/restart_data_generation"

os.environ['MLFLOW_S3_ENDPOINT_URL'] = "http://10.43.101.168:30900"
os.environ['AWS_ACCESS_KEY_ID'] = "minioadmin"
os.environ['AWS_SECRET_ACCESS_KEY'] = "minioadmin123"

mlflow.set_tracking_uri(IP_MLFLOW)

In [8]:
def split_data():
    #cambiar por postgresHook de Airflow
    conn = psycopg2.connect(
        "host=10.43.101.168 port=30543 dbname=airflow user=airflow password=airflowpass"
    )
    cursor = conn.cursor()
    
    # 1. Delete table if exists
    cursor.execute(f"DROP TABLE IF EXISTS train_data;")
    cursor.execute(f"DROP TABLE IF EXISTS test_data;")
    print(f"Tables dropped")

    create_table_query = f"""
    CREATE TABLE train_data (
        brokered_by FLOAT,
        price FLOAT,
        bed FLOAT,
        bath FLOAT,
        acre_lot FLOAT,
        street FLOAT,
        zip_code FLOAT,
        house_size FLOAT);
    """
    cursor.execute(create_table_query)

    create_table_query = f"""
    CREATE TABLE test_data (
        brokered_by FLOAT,
        price FLOAT,
        bed FLOAT,
        bath FLOAT,
        acre_lot FLOAT,
        street FLOAT,
        zip_code FLOAT,
        house_size FLOAT);
    """
    cursor.execute(create_table_query)

    print(f"Tables created successfully")

    params = {"group_number": 3, "day": "Tuesday"}
    try:
        response = requests.get(API_URL, params=params, timeout=60)
        response.raise_for_status()
        
        json_data = response.json()
        tabla = json_data.get("data", [])
    
    except requests.exceptions.HTTPError:
        # Try to get error details from JSON response
    
        try:
            response = requests.get(REGEN_URL, params=params, timeout=60)
            response.raise_for_status()
            response = requests.get(API_URL, params=params, timeout=60)
            response.raise_for_status()
            
            json_data = response.json()
            tabla = json_data.get("data", [])
    
        except requests.exceptions.HTTPError as http_err:
            error_data = response.json()
            print(f"API Error: {error_data.get('message', 'No error message provided')}")
            print(f"Status Code: {response.status_code}")
            if 'details' in error_data:
                print(f"Details: {error_data['details']}")

    df = pd.DataFrame(tabla).drop(['status','city','state','prev_sold_date'], axis=1)#.iloc[0:5003]
    df_revuelta = df.sample(frac=0.001).reset_index(drop=True)
    print(f'Filas en dataframe: {df.shape[0]}')
    df_train = df_revuelta.iloc[0:int(df_revuelta.shape[0]*0.8)]
    df_test = df_revuelta.iloc[int(df_revuelta.shape[0]*0.8):]
    print('Datos leidos y distribuidos')

    for index,data in df_train.iterrows():
        insert_query = f"""
        INSERT INTO train_data (brokered_by, price,	bed, bath, acre_lot, street, zip_code, house_size)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """
        cursor.execute(insert_query, (
            data['brokered_by'],
            data['price'],
            data['bed'],
            data['bath'],
            data['acre_lot'],
            data['street'],
            data['zip_code'],
            data['house_size'],
        ))

    for index,data in df_test.iterrows():
        insert_query = f"""
        INSERT INTO test_data (brokered_by, price,	bed, bath, acre_lot, street, zip_code, house_size)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """
        cursor.execute(insert_query, (
            data['brokered_by'],
            data['price'],
            data['bed'],
            data['bath'],
            data['acre_lot'],
            data['street'],
            data['zip_code'],
            data['house_size'],
        ))
    conn.commit()
    print('Tables loaded with data')

    conn.close()

    

In [9]:
split_data()

Tables dropped
Tables created successfully
Filas en dataframe: 5003
Datos leidos y distribuidos
Tables loaded with data


In [6]:
def entrenamiento():

    conn = psycopg2.connect(
        "host=10.43.101.168 port=30543 dbname=airflow user=airflow password=airflowpass"
    )
    # Read SQL query into a DataFrame
    df_train = pd.read_sql("SELECT * FROM train_data;", conn)
    conn.close()
    
    client = mlflow.tracking.MlflowClient()
    
    experiment_name = "argocd_experiment"
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        mlflow.create_experiment(experiment_name)
    mlflow.set_experiment(experiment_name)
    
    with mlflow.start_run(run_name="svm_training") as run:
    # inicializar svm
            svm = SVC()
            model_name = "svm-model"
            
            X = df_train.drop(['price'],axis=1)#.iloc[0:2000]
            y = df_train['price']#.iloc[0:2000]
            
                    # buscar hiperparametros mas optimos
            print('Iniciando entrenamiento')
            svm = SVC(C=1.0, kernel='rbf', gamma='scale', probability=True)  # Ajusta los valores si lo deseas
            svm.fit(X, y)
            print('Entrenamiento finalizado')
    
            # mlflow.set_tag("column_names", ",".join(columns))
            mlflow.sklearn.log_model(
                sk_model=svm,
                artifact_path="svm",
                registered_model_name=model_name
            )
    
            latest_model_versions = client.search_model_versions(f"name='{model_name}'")
            latest_version = max(int(m.version) for m in latest_model_versions)  # Get the highest version
    
            print(f'Ultima versión: {latest_version}')

In [10]:
entrenamiento()

/tmp/ipykernel_539/477119990.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_train = pd.read_sql("SELECT * FROM train_data", conn)


Iniciando entrenamiento
Entrenamiento finalizado


2025/05/29 00:54:39 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/05/29 00:54:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'svm-model'.
Created version '1' of model 'svm-model'.


Ultima versión: 1


In [37]:
def validacion():
    model_name = "svm-model"

    conn = psycopg2.connect(
        "host=10.43.101.168 port=30543 dbname=airflow user=airflow password=airflowpass"
    )
    # Read SQL query into a DataFrame
    df_test = pd.read_sql("SELECT * FROM test_data;", conn)
    conn.close()
    
    client = mlflow.tracking.MlflowClient()
    
    experiment_name = "argocd_experiment"
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        mlflow.create_experiment(experiment_name)
    mlflow.set_experiment(experiment_name)
    
    with mlflow.start_run(run_name="svm_validation") as run:
    # inicializar svm

        X = df_test.drop(['price'],axis=1)#.iloc[0:2000]
        y = df_test['price']#.iloc[0:2000]

        latest_model_versions = client.search_model_versions(f"name='{model_name}'")
        latest_version = max(int(m.version) for m in latest_model_versions)  # Get the highest version

        model = mlflow.sklearn.load_model(
            model_uri=f"models:/{model_name}/latest"
        )            
            
        y_pred = model.predict(X)
        y_pred_proba = model.predict_proba(X)[:, 1]
        
        # Calcular m�tricas
        results = {
            "accuracy_latest": accuracy_score(y, y_pred),
            "mae_latest": mean_absolute_error(y, y_pred),
            "r2_latest": r2_score(y, y_pred),
            #"f1_score_latest": f1_score(y, y_pred, pos_label="YES"),
            #"roc_auc_latest": roc_auc_score(y == "YES", y_pred_proba)
        }

        mlflow.log_metrics(results)
        
        # Mostrar resultados
        print("Resultados en validaci�n:")
        for metric, value in results.items():
            print(f"- {metric}: {value:.4f}")

        try:
            model_prod = mlflow.sklearn.load_model(
                model_uri=f"models:/{model_name}/Production"
            )
            y_pred = model_prod.predict(X)
            y_pred_proba = model_prod.predict_proba(X)[:, 1]
            
            # Calcular m�tricas
            results_prod = {
                "accuracy_production": accuracy_score(y, y_pred),
                "mae_production": mean_absolute_error(y, y_pred),
                "r2_production": r2_score(y, y_pred),
            }

            mlflow.log_metrics(results_prod)

            if results['r2_latest'] >= results_prod['r2_production']:

                print('Model entrenado posee mejor desempeño que modelo en producción')
                print('Iniciando cargue en producción')

            # Transicionar a producción
                client.transition_model_version_stage(
                    name=model_name,
                    version=latest_version,
                    stage="Production"
                )
                mlflow.log_text(f"El modelo obtuvo un r2 score de {results['r2_latest']} siendo el mas alto hasta el momento y por eso sera pasado a producción",
                                f"annotations/log.txt")
            else:
                mlflow.log_text(f"El modelo obtuvo un r2 score de {results['r2_latest']} y no sobrepasó el desempeño del modelo en producción",
                                f"annotations/log.txt")
                print('Model entrenado no posee mejor desempeño que modelo en producción y sera omitido')
            
        except mlflow.exceptions.MlflowException:

            print('No se encuentra modelo en production, cargando modelo')

        # Transicionar a producción
            client.transition_model_version_stage(
                name=model_name,
                version=latest_version,
                stage="Production"
            )
            mlflow.log_text("Este modelo es el primero en ser cargado a producción",
                            f"annotations/log.txt")
            

In [38]:
#split_data()
#entrenamiento()
validacion()

/tmp/ipykernel_539/3696037734.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_test = pd.read_sql("SELECT * FROM test_data;", conn)
/jupyter_app/.venv/lib/python3.12/site-packages/mlflow/store/artifact/utils/models.py:31: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


Resultados en validaci�n:
- accuracy_latest: 0.0592
- mae_latest: 63147.4957
- r2_latest: -1.3680


/jupyter_app/.venv/lib/python3.12/site-packages/mlflow/store/artifact/utils/models.py:31: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])
2025/05/29 01:42:47 INFO mlflow.tracking._tracking_service.client: 🏃 View run svm_validation at: http://10.43.101.168:30500/#/experiments/1/runs/44e84ca463ff42e8a6a1217f20691a30.
2025/05/29 01:42:47 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://10.43.101.168:30500/#/experiments/1.


Model entrenado no posee mejor desempeño que modelo en producción y sera omitido


In [ ]:
## Snippet para que osquitar pueda leer el log del ultimo run de validación

In [39]:

    client = mlflow.tracking.MlflowClient()

# Get experiment by name
    experiment = client.get_experiment_by_name('argocd_experiment')
    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' not found")
    
# Search runs with the name filter
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.`mlflow.runName` = 'svm_validation'",
        order_by=["attributes.start_time DESC"],
        max_results=1
    )
    
    if not runs:
        print(f"No runs found with name containing '{run_name_pattern}'")

    latest_run = runs[0]
    run_id = latest_run.info.run_id
    
    # Get annotation if exists (adjust path as needed)
    try:
        annotation_content = mlflow.artifacts.load_text(
            f"runs:/{run_id}/annotations/log.txt"
        )
        print(annotation_content)
    except Exception as e:
        print(f"Annotation not found: {str(e)}")



El modelo obtuvo un r2 score de -1.368019721779799 y no sobrepasó el desempeño del modelo en producción
